# 01: NIfTI Exploration, Spatial Affine Transforms, and Interactive 3D Slicing

This notebook is the interactive guide to **3D volumetric medical imaging**:
1. **The NIfTI Format (`.nii.gz`)**: Headers, voxel resolutions ($dx, dy, dz$), and coordinate spaces.
2. **The $4 \times 4$ Affine Transformation Matrix**: Interactive mapping from discrete voxel indices $(i, j, k)$ to scanner physical world space $(x, y, z)$ in millimeters.
3. **Multi-Parametric Brain MRI Modalities**: FLAIR, T1, T1ce, and T2 contrast differences.
4. **Interactive Multi-Planar Orthogonal Slicer**: Real-time interactive sliders across Axial, Coronal, and Sagittal views with crosshairs and tumor sub-region masks.
5. **Interactive 3D Tumor Mesh Surface Reconstruction**: Rotatable, zoomable 3D Plotly surface model of the brain tumor.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Ensure project src is in path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.io_utils import (
    load_nifti,
    inspect_nifti_metadata,
    voxel_to_world,
    world_to_voxel,
    print_metadata_summary
)
from src.visualization import (
    plot_orthogonal_slices,
    plot_multimodal_comparison,
    interactive_orthogonal_viewer,
    interactive_affine_calculator,
    plot_3d_tumor_mesh_plotly,
    BRATS_LABEL_NAMES
)

print("Modules and interactive visualization tools loaded successfully!")

---
## 1. Interactive Subject Selector
Browse any subject from the downloaded **BraTS dataset** and dynamically inspect its dimensions and NIfTI header metadata.

In [ ]:
DATA_DIR = PROJECT_ROOT / "data" / "Task01_BrainTumour"
images_tr_dir = DATA_DIR / "imagesTr"
labels_tr_dir = DATA_DIR / "labelsTr"

subject_files = sorted([f.name for f in images_tr_dir.glob("*.nii.gz")])
print(f"Total available subjects in dataset: {len(subject_files)}")

# Interactive subject dropdown
subject_dropdown = widgets.Dropdown(
    options=subject_files,
    value=subject_files[0] if subject_files else None,
    description='Subject ID:'
)

def on_subject_select(subject_name):
    img_path = images_tr_dir / subject_name
    lbl_path = labels_tr_dir / subject_name
    
    metadata = inspect_nifti_metadata(img_path)
    print_metadata_summary(metadata)

widgets.interactive(on_subject_select, subject_name=subject_dropdown)

---
## 2. Deep Dive: Mathematical Affine Coordinate Transformation
The $4 \times 4$ Affine matrix $M$ connects discrete voxel indices $(i, j, k)$ to the patient's continuous physical scanner space $(x, y, z)$ in millimeters:

$$\begin{bmatrix} x \\ y \\ z \\ 1 \end{bmatrix} = \begin{bmatrix} M_{00} & M_{01} & M_{02} & M_{03} \\ M_{10} & M_{11} & M_{12} & M_{13} \\ M_{20} & M_{21} & M_{22} & M_{23} \\ 0 & 0 & 0 & 1 \end{bmatrix} \begin{bmatrix} i \\ j \\ k \\ 1 \end{bmatrix}$$

Use the interactive sliders below to see how voxel coordinates $(i, j, k)$ transform into real-world coordinates in real time:

In [ ]:
# Load the selected subject volume and affine matrix
selected_subject = subject_dropdown.value or subject_files[0]
img_data, affine_matrix, header = load_nifti(images_tr_dir / selected_subject)
lbl_data, _, _ = load_nifti(labels_tr_dir / selected_subject)

# Launch interactive affine coordinate calculator
interactive_affine_calculator(affine_matrix, img_data.shape)

---
## 3. Interactive Multi-Planar Orthogonal Slice Viewer
Explore the 3D volume across all three anatomical planes (**Sagittal**, **Coronal**, **Axial**).

**Interactive Controls:**
* **Sliders (X, Y, Z)**: Navigate 3D cross-sections.
* **Modality Dropdown**: Switch between `0: FLAIR`, `1: T1`, `2: T1ce (Contrast-Enhanced)`, and `3: T2`.
* **Show Mask & Opacity**: Toggle and adjust the transparency of the ground-truth tumor segmentation.
* **Crosshairs**: Synchronized intersection lines across all three orthogonal views.

In [ ]:
modality_names = [
    "0: FLAIR (Peritumoral Edema)",
    "1: T1 (Anatomical Structure)",
    "2: T1ce (Active Enhancing Core)",
    "3: T2 (Tumor Boundary)"
]

interactive_orthogonal_viewer(
    volume=img_data,
    mask=lbl_data,
    modality_names=modality_names
)

---
## 4. Multi-Modal MRI Contrast Comparison
Let's compare all 4 MRI modalities side-by-side on an axial slice containing the largest tumor cross-section.

In [ ]:
# Find axial slice with maximum tumor content
tumor_voxels_per_slice = [np.sum(lbl_data[:, :, z] > 0) for z in range(lbl_data.shape[2])]
peak_tumor_z = int(np.argmax(tumor_voxels_per_slice))
print(f"Slice with highest tumor volume: Axial Z = {peak_tumor_z}")

fig = plot_multimodal_comparison(
    volume_4d=img_data,
    slice_z=peak_tumor_z,
    modality_names=modality_names,
    mask=lbl_data,
    title=f"Multi-Parametric MRI Comparison ({selected_subject} | Axial Z={peak_tumor_z})",
    figsize=(18, 4.5)
)
plt.show()

---
## 5. Interactive 3D Tumor Surface Mesh Reconstruction (Plotly)
Using **Marching Cubes**, we extract the 3D polygonal isosurfaces for each tumor sub-compartment:
* 🔴 **Necrotic Core (NCR)** (Label 1) - Central dying tissue
* 🟢 **Peritumoral Edema (ED)** (Label 2) - Fluid swelling surrounding the tumor
* 🟡 **Enhancing Tumor (ET)** (Label 3) - Actively growing, vascularized rim

*Click and drag to rotate, zoom, and inspect the 3D tumor geometry:*

In [ ]:
fig_3d = plot_3d_tumor_mesh_plotly(
    mask=lbl_data,
    affine=affine_matrix,
    step_size=2,
    title=f"3D Interactive Brain Tumor Reconstruction ({selected_subject})"
)
if fig_3d:
    fig_3d.show()

---
### Key Takeaways
1. **Volumetric Foundation for Tractography**: Before fibers can be classified in 3D (Project 3), their spatial coordinates rely on the volumetric NIfTI voxel grid and affine transforms mastered here.
2. **Multi-Parametric Complementarity**: No single MRI modality shows the entire tumor; combining FLAIR, T1, T1ce, and T2 is essential for multi-class segmentation.
3. In **Notebook 2**, we build and train a **3D U-Net** with MONAI to segment these volumes automatically!